# The number in the deck, six months later

Somebody asks where a figure came from. The notebook that produced it has been edited twice.
The data file it read has been re-exported. The person who ran it has changed teams. What
usually survives is a `results.pkl` that loads under a different pandas version, or does not.

An `Analysis` is the frozen container a notebook holds — specs by role, the panel, the
posterior, evidence records, the assumption ledger, and provenance — and `save_analysis`
writes it as a directory of plain text and arrays:

```
analysis.axiom/
├── manifest.json        format version, axiom version, hashes, declared bases and units
├── specs/<role>.json    one envelope per Spec
├── panel.csv            the Panel (17 significant digits) + panel_roles.json
├── posterior.npz        draws + JSON meta
└── evidence/            evidence.jsonl, ledger.jsonl
```

**No pickle anywhere.** Loading replays specs by class name and rehydrates arrays: a saved
analysis cannot execute code, cannot depend on the version of a library that wrote it, and
can be read by a human with `cat` when all else fails.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from axiom.core import BASES, D, LedgerLine, Outcome, Posterior, TimeWindow, Treatment
from axiom.data import Panel, RoleMap, fit_scaling
from axiom.io import (
    FORMAT_VERSION,
    Analysis,
    ArtifactRegistry,
    FormatError,
    Provenance,
    environment_fingerprint,
    load_analysis,
    save_analysis,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, heat

enable();  # every axiom result renders itself from here on

In [ ]:
rng = np.random.default_rng(3)
df = pd.DataFrame({"u": np.repeat(["01", "02", "03"], 8), "t": np.tile(range(8), 3),
                   "y": rng.gamma(3, 10, 24), "x": rng.uniform(0, 100, 24)})
roles = RoleMap(unit="u", time="t",
                outcome=("y", Outcome(name="y_total", dimension=D.outcome)),
                treatments={"x": Treatment(name="x_dose", dimension=D.currency, unit="USD")})
panel = Panel(df, roles)
posterior = Posterior({"beta": rng.normal(0.5, 0.1, size=(2, 200))}, provenance={"seed": 3})

## Build the container

`Analysis` is immutable; `with_*` methods return a new one. `hashes()` is the set of content
hashes that becomes the provenance record — the answer to "was this the same data?" that
does not depend on anybody's memory of which file was current.

In [ ]:
analysis = (
    Analysis(specs={"roles": roles, "scaling": fit_scaling(panel), "window": TimeWindow(start=0, stop=8)})
    .with_panel(panel)
    .with_posterior(posterior)
    .with_ledger_line(LedgerLine(kind="note", statement="toy analysis for the io notebook"))
)
print(analysis.summary())
table(
    [[k, v[:16]] for k, v in analysis.hashes().items()],
    headers=("object", "content hash"),
)

## Save

`save_analysis` refuses to overwrite unless told to, stamps provenance, and returns the
stamped `Analysis` — which compares equal to what `load_analysis` returns.

In [ ]:
root = Path(tempfile.mkdtemp()) / "toy.axiom"
saved = save_analysis(analysis, root, seed=3)
print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()))
print(FORMAT_VERSION, "|", saved.provenance.seed, saved.provenance.created)

In [ ]:
loaded = load_analysis(root)
print("equal after round-trip:", loaded == saved)
print(loaded.spec("window"), "|", loaded.panel, "|", loaded.posterior)
print(loaded.ledger[0].statement)

## Three runs on the shelf, and the question that gets asked about them

Below: three saved analyses, compared artifact by artifact. The middle one refit the same
panel; the last one refit a panel with one plot dropped. Six months later, that distinction
is the entire meaning of "the results changed" — and it is a hash comparison, not a
recollection.

In [ ]:
dropped = Panel(df[df["u"] != "03"], roles)
runs = {
    "run 1 · baseline": analysis,
    "run 2 · refit, same data": analysis.with_posterior(
        Posterior({"beta": rng.normal(0.55, 0.1, size=(2, 200))}, provenance={"seed": 4})
    ),
    "run 3 · one unit dropped": analysis.with_panel(dropped).with_spec("scaling", fit_scaling(dropped)),
}
artifacts = ["roles", "scaling", "window", "panel", "posterior"]
base = analysis.hashes()
grid = [[float(run.hashes().get(a) == base.get(a)) for a in artifacts] for run in runs.values()]

fig = heat(
    grid, artifacts, list(runs),
    text_fmt="{:.0f}",
    colorbar_title="same as run 1",
    title="What actually changed between these runs",
    subtitle="content hash of every stored artifact, against the baseline",
    height=300,
)
caption(fig, "Run 2 differs only in its posterior — a reseed. Run 3 differs in the panel and "
             "in the scaling that was fit to it, which is why its numbers moved. Neither "
             "answer required opening a notebook.")

## Provenance and the environment

`Provenance` records the axiom version, a UTC timestamp, the content hashes, the seed, and the
environment fingerprint. You can build one explicitly (useful for deterministic tests).

In [ ]:
print(environment_fingerprint())
prov = Provenance(axiom_version="0.0.0", created="2026-08-21T00:00:00+00:00", hashes=analysis.hashes(), seed=3)
print(prov.content_hash()[:16])

## Declared bases travel with the analysis

If a domain declared extra base dimensions, the manifest records them and `load_analysis`
re-declares them before replaying specs (note 0002.6). Without that, an agronomy analysis
would load in a fresh process and fail to reconstruct its own units — the dimension system
of `nbs/core/01-dimensions.ipynb` is only useful if it survives a save.

In [ ]:
import json

BASES.declare("mass", symbol="M")
root2 = root.parent / "with_mass.axiom"
save_analysis(Analysis(specs={"m": Treatment(name="seed_mass", dimension=D.mass, unit="kg")}), root2)
print(json.loads((root2 / "manifest.json").read_text())["bases"])

## Refusals

Overwriting without `overwrite=True`, a directory that is not an analysis, an unknown format
version, and a panel whose bytes no longer match their hash are all errors. The last one is
the important one: an analysis that was edited on disk after it was saved is not the analysis
that produced the figure, and it says so rather than loading.

In [ ]:
try:
    save_analysis(analysis, root)
except FileExistsError as e:
    print("FileExistsError:", e)

m = root / "manifest.json"
m.write_text(m.read_text().replace(f'"format_version": "{FORMAT_VERSION}"', '"format_version": "99"'))
try:
    load_analysis(root)
except FormatError as e:
    print("FormatError:", e)

## A content-addressed registry

`ArtifactRegistry` stores specs by hash. `put` is idempotent; `get` verifies the hash on read.
Storing by content rather than by filename means the same spec saved by two people is one
entry, and a corrupted read is caught at the point of reading rather than in a result.

In [ ]:
reg = ArtifactRegistry(root.parent / "registry")
h = reg.put(roles)
print(h[:16], reg.put(roles) == h, len(reg), h in reg)
print(reg.get(h) == roles)
print(list(reg))

## What this bought you

A result you can hand to somebody else — with the data, the configuration, the draws, the
ledger of assumptions, and the seed — in a format that will still open when the library
version has moved on, and that tells you if it was touched.